# SUMMARY: STEP 5 - FEATURE ENGINEERING & DATA PREPARATION

This notebook covers the processing and preparation of session and network data:

1. Verification: Validate SHA-256 hashes, timestamps, and session IDs.
2. Feature Extraction: Build behavioral, network, and geo-spatial features.
3. Data Cleaning: Clean missing values and prevent data leakage.
4. Export: Perform final checks and export the clean dataset for training.

In [ ]:
import os
import hashlib

SOURCE = "/content/session_replay_v1.csv"
EXPECTED_SHA = "7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846"

print("=" * 70)
print("STEP 5A — REPLAY FILE COMPLETION CHECK")
print("=" * 70)

if not os.path.exists(SOURCE):
    raise FileNotFoundError("STOP — replay file is not uploaded yet.")

size = os.path.getsize(SOURCE)

print(f"File exists: PASS")
print(f"Current size: {size:,} bytes")
print("Calculating SHA-256...")

sha = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

actual_sha = sha.hexdigest()

print(f"Actual SHA-256: {actual_sha}")
print(f"Expected SHA-256: {EXPECTED_SHA}")

if actual_sha != EXPECTED_SHA:
    print("\nSTATUS: WAIT")
    print("The upload is NOT verified yet.")
    print("DO NOT continue to Step 5B.")
else:
    print("\nSTATUS: PASS")
    print("Replay checkpoint verified successfully.")
    print("SAFE TO CONTINUE TO STEP 5B.")

STEP 5A — REPLAY FILE COMPLETION CHECK
File exists: PASS
Current size: 16,685,010 bytes
Calculating SHA-256...
Actual SHA-256: 7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Expected SHA-256: 7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846

STATUS: PASS
Replay checkpoint verified successfully.
SAFE TO CONTINUE TO STEP 5B.


In [ ]:
import os
import hashlib
import pandas as pd

SOURCE = "/content/session_replay_v1.csv"

EXPECTED_SHA = (
    "7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846"
)

print("=" * 70)
print("STEP 5B-2 — SESSION ID DIAGNOSTIC")
print("=" * 70)

# --------------------------------------------------
# 1. FILE CHECK
# --------------------------------------------------
if not os.path.exists(SOURCE):
    raise FileNotFoundError(f"STOP — file not found:\n{SOURCE}")

print("\n[1] SOURCE")
print("Exists: PASS")
print(f"Path: {SOURCE}")
print(f"Size: {os.path.getsize(SOURCE):,} bytes")

# --------------------------------------------------
# 2. SHA CHECK
# --------------------------------------------------
print("\n[2] SHA-256")

sha = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

actual_sha = sha.hexdigest()

print(f"Expected: {EXPECTED_SHA}")
print(f"Actual:   {actual_sha}")

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — SHA-256 mismatch.\n"
        "Do NOT continue."
    )

print("Verification: PASS")

# --------------------------------------------------
# 3. LOAD
# --------------------------------------------------
print("\n[3] LOAD")

df = pd.read_csv(SOURCE, low_memory=False)

print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

if len(df) != 58895:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)}"
    )

print("Row count: PASS")

# --------------------------------------------------
# 4. SESSION ID CHECK
# --------------------------------------------------
print("\n[4] SESSION IDENTIFIERS")

for col in ["raw_session_id", "reconstructed_session_id"]:
    if col not in df.columns:
        raise ValueError(f"STOP — missing column: {col}")

    missing = int(df[col].isna().sum())
    empty = int(
        df[col]
        .astype("string")
        .str.strip()
        .eq("")
        .sum()
    )

    unique = df[col].nunique(dropna=True)

    print(f"\n{col}")
    print(f"  Missing: {missing}")
    print(f"  Empty:   {empty}")
    print(f"  Unique:  {unique:,}")

# --------------------------------------------------
# 5. DUPLICATE / TIMESTAMP CHECK
# --------------------------------------------------
print("\n[5] TIMESTAMP")

df["timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_ts = int(df["timestamp"].isna().sum())

print(f"Invalid timestamps: {invalid_ts}")

if invalid_ts != 0:
    raise ValueError(
        f"STOP — invalid timestamps: {invalid_ts}"
    )

print("Timestamp validation: PASS")

# --------------------------------------------------
# 6. REPLAY STRUCTURE
# --------------------------------------------------
print("\n[6] REPLAY STRUCTURE")

required = [
    "reconstructed_session_id",
    "raw_session_id",
    "timestamp",
    "replay_event_index",
    "event_sequence",
    "session_event_count",
    "session_start_time",
    "session_end_time",
    "session_duration_seconds",
    "inter_event_gap_seconds",
]

missing_required = [
    c for c in required
    if c not in df.columns
]

print(f"Missing required columns: {missing_required}")

if missing_required:
    raise ValueError(
        f"STOP — missing replay columns: {missing_required}"
    )

print("Required replay structure: PASS")

# --------------------------------------------------
# 7. FINAL DIAGNOSTIC
# --------------------------------------------------
raw_missing = int(df["raw_session_id"].isna().sum())
recon_missing = int(df["reconstructed_session_id"].isna().sum())

print("\n" + "=" * 70)
print("STEP 5B-2 RESULT")
print("=" * 70)

print(f"Rows:                         {len(df):,}")
print(f"raw_session_id missing:      {raw_missing}")
print(f"reconstructed_session_id missing: {recon_missing}")
print(f"Timestamp invalid:            {invalid_ts}")
print(f"SHA verification:             PASS")

if raw_missing == 0 and recon_missing == 0 and invalid_ts == 0:
    print("\nSTATUS: PASS")
    print("SAFE TO CONTINUE TO STEP 5B BEHAVIORAL FEATURES")
else:
    print("\nSTATUS: STOP")
    print("Do NOT modify the Replay dataset.")
    print("Send me this output before continuing.")

STEP 5B-2 — SESSION ID DIAGNOSTIC

[1] SOURCE
Exists: PASS
Path: /content/session_replay_v1.csv
Size: 16,685,010 bytes

[2] SHA-256
Expected: 7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Actual:   7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Verification: PASS

[3] LOAD
Shape: (58895, 32)
Rows: 58,895
Columns: 32
Row count: PASS

[4] SESSION IDENTIFIERS

raw_session_id
  Missing: 75
  Empty:   0
  Unique:  11,879

reconstructed_session_id
  Missing: 0
  Empty:   0
  Unique:  11,880

[5] TIMESTAMP
Invalid timestamps: 0
Timestamp validation: PASS

[6] REPLAY STRUCTURE
Missing required columns: []
Required replay structure: PASS

STEP 5B-2 RESULT
Rows:                         58,895
raw_session_id missing:      75
reconstructed_session_id missing: 0
Timestamp invalid:            0
SHA verification:             PASS

STATUS: STOP
Do NOT modify the Replay dataset.
Send me this output before continuing.


In [ ]:
import os
import hashlib
import pandas as pd

SOURCE = "/content/session_replay_v1.csv"
EXPECTED_SHA = "7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846"

print("=" * 70)
print("STEP 5B-3 — RAW SESSION ID MISSING-ROW DIAGNOSTIC")
print("=" * 70)

# ---------------------------------------------------------
# 1. Verify file exists
# ---------------------------------------------------------
if not os.path.exists(SOURCE):
    raise FileNotFoundError(f"STOP — Replay file not found:\n{SOURCE}")

print(f"[1] SOURCE\nExists: PASS\nPath: {SOURCE}")

# ---------------------------------------------------------
# 2. Verify SHA before reading
# ---------------------------------------------------------
sha = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha.update(chunk)

actual_sha = sha.hexdigest()

print("\n[2] SHA-256")
print(f"Expected: {EXPECTED_SHA}")
print(f"Actual:   {actual_sha}")

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — SHA-256 mismatch.\n"
        "Do NOT modify or continue with this Replay file."
    )

print("Verification: PASS")

# ---------------------------------------------------------
# 3. Load read-only
# ---------------------------------------------------------
df = pd.read_csv(SOURCE, low_memory=False)

print("\n[3] DATASET")
print(f"Shape: {df.shape}")
print(f"Rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

if len(df) != 58895:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)}"
    )

# ---------------------------------------------------------
# 4. Identify missing raw_session_id rows
# ---------------------------------------------------------
raw_missing = (
    df["raw_session_id"].isna()
    | df["raw_session_id"].astype("string").str.strip().eq("")
)

missing_rows = df.loc[
    raw_missing,
    [
        "reconstructed_session_id",
        "raw_session_id",
        "timestamp",
        "replay_event_index",
        "event_sequence",
        "session_event_count",
        "session_start_time",
        "session_end_time",
        "session_duration_seconds",
        "inter_event_gap_seconds",
        "source_ip",
        "destination_ip",
        "protocol",
        "honeypot",
    ]
].copy()

print("\n[4] RAW SESSION ID")
print(f"Missing raw_session_id rows: {len(missing_rows):,}")

# ---------------------------------------------------------
# 5. Check reconstructed IDs of affected rows
# ---------------------------------------------------------
print("\n[5] AFFECTED RECONSTRUCTED SESSIONS")

affected_reconstructed = (
    missing_rows["reconstructed_session_id"]
    .dropna()
    .astype(str)
    .unique()
)

print(
    f"Unique reconstructed_session_id affected: "
    f"{len(affected_reconstructed):,}"
)

# ---------------------------------------------------------
# 6. Check whether each affected reconstructed session
#    contains other rows with a raw_session_id
# ---------------------------------------------------------
affected_all = df[
    df["reconstructed_session_id"]
    .astype(str)
    .isin(affected_reconstructed.astype(str))
].copy()

affected_all["raw_missing"] = (
    affected_all["raw_session_id"].isna()
    | affected_all["raw_session_id"]
        .astype("string")
        .str.strip()
        .eq("")
)

session_diagnostic = (
    affected_all
    .groupby("reconstructed_session_id", dropna=False)
    .agg(
        total_rows=("reconstructed_session_id", "size"),
        missing_raw_session_id=("raw_missing", "sum"),
        nonmissing_raw_session_id=(
            "raw_session_id",
            lambda s: s.notna().sum()
        ),
        unique_raw_session_ids=(
            "raw_session_id",
            lambda s: s.dropna().astype(str).nunique()
        ),
        start_time=("timestamp", "min"),
        end_time=("timestamp", "max"),
    )
    .reset_index()
)

print(
    f"\nAffected reconstructed sessions: "
    f"{len(session_diagnostic):,}"
)

print("\nDiagnostic summary:")
print(session_diagnostic.to_string(index=False))

# ---------------------------------------------------------
# 7. Determine whether missing raw IDs are isolated
#    or entire reconstructed sessions
# ---------------------------------------------------------
entire_session_missing = session_diagnostic[
    session_diagnostic["nonmissing_raw_session_id"] == 0
]

partial_session_missing = session_diagnostic[
    session_diagnostic["nonmissing_raw_session_id"] > 0
]

print("\n[6] CLASSIFICATION")
print(
    f"Entire reconstructed session has no raw_session_id: "
    f"{len(entire_session_missing):,}"
)

print(
    f"Partial reconstructed session has missing raw_session_id: "
    f"{len(partial_session_missing):,}"
)

# ---------------------------------------------------------
# 8. Check whether reconstructed IDs are duplicated
#    across rows with/without raw IDs
# ---------------------------------------------------------
print("\n[7] RECONSTRUCTED SESSION INTEGRITY")

print(
    f"Unique reconstructed sessions overall: "
    f"{df['reconstructed_session_id'].nunique(dropna=True):,}"
)

print(
    f"Rows with reconstructed_session_id missing: "
    f"{df['reconstructed_session_id'].isna().sum():,}"
)

# ---------------------------------------------------------
# 9. Save diagnostic ONLY
#    Do NOT overwrite Replay dataset
# ---------------------------------------------------------
DIAGNOSTIC_OUT = "/content/session_replay_v1_raw_session_id_diagnostic.csv"

missing_rows.to_csv(
    DIAGNOSTIC_OUT,
    index=False
)

print("\n[8] DIAGNOSTIC OUTPUT")
print(f"Saved: {DIAGNOSTIC_OUT}")
print(f"Rows saved: {len(missing_rows):,}")

# ---------------------------------------------------------
# 10. Final safety check
# ---------------------------------------------------------
sha_after = hashlib.sha256()

with open(SOURCE, "rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        sha_after.update(chunk)

sha_after = sha_after.hexdigest()

print("\n[9] SOURCE PROTECTION")
print(f"SHA before: {actual_sha}")
print(f"SHA after:  {sha_after}")
print(f"Source unchanged: {'PASS' if actual_sha == sha_after else 'FAIL'}")

print("\n" + "=" * 70)
print("STATUS: DIAGNOSTIC COMPLETE")
print("Replay dataset was NOT modified.")
print("DO NOT proceed to feature engineering yet.")
print("=" * 70)

STEP 5B-3 — RAW SESSION ID MISSING-ROW DIAGNOSTIC
[1] SOURCE
Exists: PASS
Path: /content/session_replay_v1.csv

[2] SHA-256
Expected: 7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Actual:   7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Verification: PASS

[3] DATASET
Shape: (58895, 32)
Rows: 58,895
Columns: 32

[4] RAW SESSION ID
Missing raw_session_id rows: 75

[5] AFFECTED RECONSTRUCTED SESSIONS
Unique reconstructed_session_id affected: 1

Affected reconstructed sessions: 1

Diagnostic summary:
reconstructed_session_id  total_rows  missing_raw_session_id  nonmissing_raw_session_id  unique_raw_session_ids                       start_time                         end_time
       reconstructed_nan          75                      75                          0                       0 2025-06-09 20:06:14.496000+00:00 2025-06-11 15:33:07.648000+00:00

[6] CLASSIFICATION
Entire reconstructed session has no raw_session_id: 1
Partial reconstructed session 

In [ ]:
# ================================================================
# STEP 5B-0 — RECONSTRUCTION CHECKPOINT CONSISTENCY AUDIT
# READ-ONLY / NO FILE MODIFICATION
# ================================================================

import os
import json
import hashlib
import pandas as pd

print("=" * 70)
print("STEP 5B-0 — RECONSTRUCTION CHECKPOINT CONSISTENCY AUDIT")
print("=" * 70)

# ------------------------------------------------
# 1. FILE PATHS
# ------------------------------------------------

RECON = "/content/session_reconstructed_v1.csv"
RECON_SHA = "/content/session_reconstructed_v1.sha256"
RECON_AUDIT = "/content/session_reconstructed_v1_audit.json"

BACKUP_MANIFEST = "/content/backup_manifest.json"
BACKUP_MANIFEST_SHA = "/content/backup_manifest.sha256"

FILES = {
    "reconstructed_dataset": RECON,
    "reconstructed_sha256": RECON_SHA,
    "reconstructed_audit": RECON_AUDIT,
    "backup_manifest": BACKUP_MANIFEST,
    "backup_manifest_sha256": BACKUP_MANIFEST_SHA,
}

print("\n[1] REQUIRED FILES")

for name, path in FILES.items():
    status = "PASS" if os.path.exists(path) else "MISSING"
    print(f"{name:28s}: {status} -> {path}")

missing_files = [
    path for path in FILES.values()
    if not os.path.exists(path)
]

if missing_files:
    raise FileNotFoundError(
        "\nSTOP — checkpoint files are incomplete.\n"
        + "\n".join(missing_files)
    )

# ------------------------------------------------
# 2. SHA-256 FUNCTION
# ------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------
# 3. READ EXPECTED SHA
# ------------------------------------------------

with open(RECON_SHA, "r", encoding="utf-8") as f:
    manifest_text = f.read().strip()

expected_sha = manifest_text.split()[0]

print("\n[2] RECONSTRUCTED DATASET SHA-256")

actual_sha = sha256_file(RECON)

print("Expected:", expected_sha)
print("Actual:  ", actual_sha)

if actual_sha != expected_sha:
    raise ValueError(
        "\nSTOP — reconstructed dataset SHA mismatch.\n"
        "DO NOT modify the dataset.\n"
        "DO NOT continue to feature engineering."
    )

print("Verification: PASS")

# ------------------------------------------------
# 4. LOAD RECONSTRUCTED DATASET
# ------------------------------------------------

print("\n[3] LOADING RECONSTRUCTED DATASET")

df = pd.read_csv(
    RECON,
    low_memory=False
)

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

# ------------------------------------------------
# 5. ROW COUNT
# ------------------------------------------------

EXPECTED_ROWS = 58895

if len(df) != EXPECTED_ROWS:
    raise ValueError(
        f"\nSTOP — unexpected row count: {len(df)} "
        f"(expected {EXPECTED_ROWS})"
    )

print("Row count: PASS")

# ------------------------------------------------
# 6. REQUIRED RECONSTRUCTION COLUMNS
# ------------------------------------------------

REQUIRED_COLUMNS = [
    "timestamp",
    "source_ip",
    "destination_ip",
    "raw_session_id",
    "reconstructed_session_id",
    "reason",
    "reconstruction_status",
    "reconstruction_action",
    "event_sequence",
]

missing_columns = [
    c for c in REQUIRED_COLUMNS
    if c not in df.columns
]

print("\n[4] REQUIRED RECONSTRUCTION COLUMNS")
print("Missing:", missing_columns)

if missing_columns:
    raise ValueError(
        f"STOP — required reconstruction columns missing: "
        f"{missing_columns}"
    )

print("Schema: PASS")

# ------------------------------------------------
# 7. TIMESTAMP VALIDATION
# ------------------------------------------------

print("\n[5] TIMESTAMP VALIDATION")

timestamps = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_timestamps = timestamps.isna().sum()

print("Invalid timestamps:", invalid_timestamps)

if invalid_timestamps != 0:
    raise ValueError(
        "STOP — invalid timestamps detected."
    )

print("Timestamp validation: PASS")

# ------------------------------------------------
# 8. SESSION COUNTS
# ------------------------------------------------

print("\n[6] SESSION COUNTS")

raw_unique = df["raw_session_id"].nunique(dropna=True)
reconstructed_unique = df["reconstructed_session_id"].nunique(dropna=True)

raw_missing = df["raw_session_id"].isna().sum()
reconstructed_missing = df["reconstructed_session_id"].isna().sum()

print("Unique raw sessions:", raw_unique)
print("Unique reconstructed sessions:", reconstructed_unique)
print("Missing raw_session_id:", raw_missing)
print("Missing reconstructed_session_id:", reconstructed_missing)

if reconstructed_missing != 0:
    raise ValueError(
        "STOP — reconstructed_session_id contains missing values."
    )

print("Reconstructed session integrity: PASS")

# ------------------------------------------------
# 9. RECONSTRUCTION STATUS / REASON
# ------------------------------------------------

print("\n[7] RECONSTRUCTION STATUS")

print(
    df["reconstruction_status"]
    .value_counts(dropna=False)
)

print("\n[8] RECONSTRUCTION REASON")

print(
    df["reason"]
    .value_counts(dropna=False)
)

# ------------------------------------------------
# 10. SAFETY CHECK
# ------------------------------------------------

print("\n[9] RECONSTRUCTION SAFETY")

if "attack_type" in df.columns:
    print("attack_type present: YES")
    print("NOTE: allowed in reconstructed source;")
    print("      it must NOT be used as a feature.")
else:
    print("attack_type present: NO")

print("Source modification: NO")
print("Automatic split performed here: NO")
print("Automatic merge performed here: NO")
print("Feature engineering performed here: NO")
print("ML features created here: NO")

# ------------------------------------------------
# 11. AUDIT JSON
# ------------------------------------------------

print("\n[10] AUDIT FILE")

with open(RECON_AUDIT, "r", encoding="utf-8") as f:
    audit = json.load(f)

print("JSON readable: PASS")
print("Audit type:", type(audit).__name__)

# ------------------------------------------------
# 12. BACKUP MANIFEST
# ------------------------------------------------

print("\n[11] BACKUP MANIFEST")

with open(BACKUP_MANIFEST, "r", encoding="utf-8") as f:
    backup_manifest = json.load(f)

print("JSON readable: PASS")
print("Manifest type:", type(backup_manifest).__name__)

# ------------------------------------------------
# 13. BACKUP MANIFEST SHA
# ------------------------------------------------

print("\n[12] BACKUP MANIFEST SHA-256")

actual_manifest_sha = sha256_file(BACKUP_MANIFEST)

with open(BACKUP_MANIFEST_SHA, "r", encoding="utf-8") as f:
    manifest_sha_text = f.read().strip()

expected_manifest_sha = manifest_sha_text.split()[0]

print("Expected:", expected_manifest_sha)
print("Actual:  ", actual_manifest_sha)

if actual_manifest_sha != expected_manifest_sha:
    raise ValueError(
        "STOP — backup manifest SHA mismatch."
    )

print("Backup manifest verification: PASS")

# ------------------------------------------------
# 14. FINAL CHECKPOINT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5B-0 — FINAL STATUS")
print("=" * 70)

print("Reconstructed dataset SHA:       PASS")
print("Rows = 58,895:                   PASS")
print("Required schema:                 PASS")
print("Timestamps:                      PASS")
print("Reconstructed session IDs:       PASS")
print("Reconstruction audit JSON:       PASS")
print("Backup manifest:                 PASS")
print("Backup manifest SHA:             PASS")
print("Source modified:                 NO")
print("Behavioral features created:     NO")
print("ML features created:             NO")

print("\nSTATUS: PASS")
print("SAFE TO CONTINUE TO STEP 5B")
print("NEXT: STEP 5B — BEHAVIORAL FEATURE CONSTRUCTION")

STEP 5B-0 — RECONSTRUCTION CHECKPOINT CONSISTENCY AUDIT

[1] REQUIRED FILES
reconstructed_dataset       : PASS -> /content/session_reconstructed_v1.csv
reconstructed_sha256        : PASS -> /content/session_reconstructed_v1.sha256
reconstructed_audit         : PASS -> /content/session_reconstructed_v1_audit.json
backup_manifest             : PASS -> /content/backup_manifest.json
backup_manifest_sha256      : PASS -> /content/backup_manifest.sha256

[2] RECONSTRUCTED DATASET SHA-256
Expected: 81afe1c709a02d3530990f2bb9695010d6ba1b17106cc0c22600b0454b929d4e
Actual:   81afe1c709a02d3530990f2bb9695010d6ba1b17106cc0c22600b0454b929d4e
Verification: PASS

[3] LOADING RECONSTRUCTED DATASET
Shape: (58895, 45)
Rows: 58895
Columns: 45
Row count: PASS

[4] REQUIRED RECONSTRUCTION COLUMNS
Missing: []
Schema: PASS

[5] TIMESTAMP VALIDATION
Invalid timestamps: 0
Timestamp validation: PASS

[6] SESSION COUNTS
Unique raw sessions: 11879
Unique reconstructed sessions: 11880
Missing raw_session_id: 75
Mi

In [ ]:
# ==============================================================
# STEP 5B-1 — BEHAVIORAL FEATURE CONSTRUCTION
# CORE SESSION-LEVEL FEATURES
# ==============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

# --------------------------------------------------------------
# [1] VERIFIED INPUT
# --------------------------------------------------------------

SOURCE = "/content/session_replay_v1.csv"

EXPECTED_SHA = (
    "7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846"
)

OUT_DIR = "/content/session_behavioral_features_v1"
os.makedirs(OUT_DIR, exist_ok=True)

OUTPUT = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v1.csv"
)

AUDIT = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v1_audit.json"
)

SHA_FILE = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v1.sha256"
)


# --------------------------------------------------------------
# [2] INPUT EXISTENCE
# --------------------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — verified Replay file not found:\n{SOURCE}"
    )

print("=" * 70)
print("STEP 5B-1 — BEHAVIORAL FEATURE CONSTRUCTION")
print("=" * 70)


# --------------------------------------------------------------
# [3] VERIFY SOURCE SHA-256
# --------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


actual_sha = sha256_file(SOURCE)

print("\n[1] SOURCE SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — Replay SHA-256 mismatch.\n"
        "The verified Step 4 Replay checkpoint must not be modified."
    )

print("Verification: PASS")


# --------------------------------------------------------------
# [4] LOAD VERIFIED REPLAY
# --------------------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if len(df) != 58895:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)} "
        f"(expected 58895)"
    )


# --------------------------------------------------------------
# [5] REQUIRED BEHAVIORAL INPUTS
# --------------------------------------------------------------

REQUIRED = [
    "reconstructed_session_id",
    "timestamp",
    "session_event_count",
    "session_duration_seconds",
    "inter_event_gap_seconds",
    "honeypot",
    "protocol",
    "source_ip",
    "destination_ip",
    "destination_port",
    "username",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "ip_reputation",
    "geoip_country_v2",
    "geoip_country_code_v2",
    "geoip_city_v2",
    "geoip_latitude_v2",
    "geoip_longitude_v2",
]

missing = [c for c in REQUIRED if c not in df.columns]

print("\n[3] REQUIRED INPUT COLUMNS")
print("Missing:", missing)

if missing:
    raise ValueError(
        f"STOP — required behavioral inputs missing: {missing}"
    )

print("Required inputs: PASS")


# --------------------------------------------------------------
# [6] LABEL / LEAKAGE PROTECTION
# --------------------------------------------------------------

LABEL_COLUMNS = [
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique",
]

for col in LABEL_COLUMNS:
    if col in df.columns:
        print(f"Label column detected in source: {col}")

print(
    "Label columns are NOT used to construct behavioral features."
)


# --------------------------------------------------------------
# [7] TIMESTAMP VALIDATION
# --------------------------------------------------------------

df["_timestamp"] = pd.to_datetime(
    df["timestamp"],
    errors="coerce",
    utc=True
)

invalid_ts = int(df["_timestamp"].isna().sum())

print("\n[4] TIMESTAMP")
print("Invalid timestamps:", invalid_ts)

if invalid_ts != 0:
    raise ValueError(
        f"STOP — invalid timestamps detected: {invalid_ts}"
    )


# --------------------------------------------------------------
# [8] SAFE NUMERIC CONVERSION
# --------------------------------------------------------------

NUMERIC_COLUMNS = [
    "session_event_count",
    "session_duration_seconds",
    "inter_event_gap_seconds",
    "user_attempt",
    "pass_attempt",
    "auth_attempt",
    "anonymous",
    "command_count",
    "ip_reputation",
    "geoip_latitude_v2",
    "geoip_longitude_v2",
]

for col in NUMERIC_COLUMNS:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )


# --------------------------------------------------------------
# [9] SESSION ORDER
# --------------------------------------------------------------

df = df.sort_values(
    ["reconstructed_session_id", "_timestamp"],
    kind="mergesort"
).reset_index(drop=True)


# --------------------------------------------------------------
# [10] SESSION-LEVEL FEATURE CONSTRUCTION
# --------------------------------------------------------------

GROUP = df.groupby(
    "reconstructed_session_id",
    sort=False,
    dropna=False
)


features = GROUP.agg(
    event_count=(
        "reconstructed_session_id",
        "size"
    ),

    session_duration_seconds=(
        "session_duration_seconds",
        "max"
    ),

    auth_attempt_total=(
        "auth_attempt",
        "sum"
    ),

    user_attempt_total=(
        "user_attempt",
        "sum"
    ),

    pass_attempt_total=(
        "pass_attempt",
        "sum"
    ),

    command_count_total=(
        "command_count",
        "sum"
    ),

    anonymous_event_count=(
        "anonymous",
        "sum"
    ),

    unique_destinations=(
        "destination_ip",
        "nunique"
    ),

    unique_destination_ports=(
        "destination_port",
        "nunique"
    ),

    unique_source_ips=(
        "source_ip",
        "nunique"
    ),

    unique_protocols=(
        "protocol",
        "nunique"
    ),

    unique_honeypots=(
        "honeypot",
        "nunique"
    ),

    unique_usernames=(
        "username",
        "nunique"
    ),

    unique_countries=(
        "geoip_country_v2",
        "nunique"
    ),

    unique_country_codes=(
        "geoip_country_code_v2",
        "nunique"
    ),

    unique_cities=(
        "geoip_city_v2",
        "nunique"
    ),

    ip_reputation_mean=(
        "ip_reputation",
        "mean"
    ),

    ip_reputation_min=(
        "ip_reputation",
        "min"
    ),

    ip_reputation_max=(
        "ip_reputation",
        "max"
    ),

    latitude_mean=(
        "geoip_latitude_v2",
        "mean"
    ),

    longitude_mean=(
        "geoip_longitude_v2",
        "mean"
    ),

    inter_event_gap_mean=(
        "inter_event_gap_seconds",
        "mean"
    ),

    inter_event_gap_min=(
        "inter_event_gap_seconds",
        "min"
    ),

    inter_event_gap_max=(
        "inter_event_gap_seconds",
        "max"
    ),
).reset_index()


# --------------------------------------------------------------
# [11] TEMPORAL BEHAVIOR
# --------------------------------------------------------------

time_features = GROUP["_timestamp"].agg(
    session_start_time="min",
    session_end_time="max"
).reset_index()

features = features.merge(
    time_features,
    on="reconstructed_session_id",
    how="left",
    validate="one_to_one"
)

features["session_duration_recomputed_seconds"] = (
    features["session_end_time"]
    - features["session_start_time"]
).dt.total_seconds()


# --------------------------------------------------------------
# [12] EVENT RATE
# --------------------------------------------------------------

features["events_per_second"] = np.where(
    features["session_duration_recomputed_seconds"] > 0,
    features["event_count"]
    / features["session_duration_recomputed_seconds"],
    0.0
)


# --------------------------------------------------------------
# [13] AUTH / COMMAND BEHAVIOR RATES
# --------------------------------------------------------------

features["auth_attempt_rate"] = np.where(
    features["session_duration_recomputed_seconds"] > 0,
    features["auth_attempt_total"]
    / features["session_duration_recomputed_seconds"],
    0.0
)

features["command_rate"] = np.where(
    features["session_duration_recomputed_seconds"] > 0,
    features["command_count_total"]
    / features["session_duration_recomputed_seconds"],
    0.0
)


# --------------------------------------------------------------
# [14] SAVE SESSION ID AS INDEX-LIKE IDENTIFIER
# --------------------------------------------------------------

features = features.sort_values(
    "reconstructed_session_id"
).reset_index(drop=True)


# --------------------------------------------------------------
# [15] FEATURE INVENTORY
# --------------------------------------------------------------

feature_columns = [
    c for c in features.columns
    if c != "reconstructed_session_id"
]

print("\n[5] FEATURE CONSTRUCTION")
print("Sessions:", len(features))
print("Behavioral features:", len(feature_columns))

print("\nFeature columns:")
for i, col in enumerate(feature_columns, 1):
    print(f"{i:02d}. {col}")


# --------------------------------------------------------------
# [16] BASIC VALIDATION
# --------------------------------------------------------------

if features["reconstructed_session_id"].isna().any():
    raise ValueError(
        "STOP — missing reconstructed_session_id in output."
    )

if features["reconstructed_session_id"].duplicated().any():
    raise ValueError(
        "STOP — duplicate reconstructed_session_id in output."
    )

if features["event_count"].sum() != len(df):
    raise ValueError(
        "STOP — feature event counts do not preserve source events."
    )

negative_duration = (
    features["session_duration_recomputed_seconds"] < 0
).sum()

if negative_duration != 0:
    raise ValueError(
        f"STOP — negative session durations: {negative_duration}"
    )


# --------------------------------------------------------------
# [17] FEATURE NUMERIC SANITY
# --------------------------------------------------------------

numeric_feature_columns = features.select_dtypes(
    include=[np.number]
).columns

nonfinite_counts = {}

for col in numeric_feature_columns:
    count = int(
        (~np.isfinite(features[col].fillna(0))).sum()
    )

    if count:
        nonfinite_counts[col] = count

if nonfinite_counts:
    raise ValueError(
        f"STOP — non-finite feature values detected: "
        f"{nonfinite_counts}"
    )


# --------------------------------------------------------------
# [18] EXPLICIT LABEL SAFETY CHECK
# --------------------------------------------------------------

for forbidden in LABEL_COLUMNS:
    if forbidden in features.columns:
        raise ValueError(
            f"STOP — forbidden label column leaked into features: "
            f"{forbidden}"
        )

print("\n[6] LABEL SAFETY")
print("attack_type in output:", "attack_type" in features.columns)
print("attack_stage in output:", "attack_stage" in features.columns)
print("severity in output:", "severity" in features.columns)
print("mitre_technique in output:", "mitre_technique" in features.columns)
print("Label leakage in constructed features: NO")


# --------------------------------------------------------------
# [19] WRITE OUTPUT
# --------------------------------------------------------------

features.to_csv(
    OUTPUT,
    index=False
)

output_sha = sha256_file(OUTPUT)

with open(SHA_FILE, "w") as f:
    f.write(f"{output_sha}  {os.path.basename(OUTPUT)}\n")


# --------------------------------------------------------------
# [20] AUDIT
# --------------------------------------------------------------

audit = {
    "step": "5B-1",
    "name": "Behavioral Feature Construction — Core Session Features",

    "source": {
        "path": SOURCE,
        "sha256": actual_sha,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
    },

    "output": {
        "path": OUTPUT,
        "sha256": output_sha,
        "sessions": int(len(features)),
        "feature_count": int(len(feature_columns)),
    },

    "event_preservation": {
        "source_events": int(len(df)),
        "feature_event_count_sum": int(features["event_count"].sum()),
        "preserved": bool(
            features["event_count"].sum() == len(df)
        ),
    },

    "label_safety": {
        "attack_type_used": False,
        "attack_stage_used": False,
        "severity_used": False,
        "mitre_technique_used": False,
        "label_leakage": False,
    },

    "source_protection": {
        "source_modified": False,
        "source_sha_preserved": True,
    },

    "feature_groups": {
        "session_volume": [
            "event_count",
        ],
        "temporal": [
            "session_duration_seconds",
            "session_duration_recomputed_seconds",
            "events_per_second",
            "inter_event_gap_mean",
            "inter_event_gap_min",
            "inter_event_gap_max",
        ],
        "authentication": [
            "auth_attempt_total",
            "user_attempt_total",
            "pass_attempt_total",
            "auth_attempt_rate",
        ],
        "command_activity": [
            "command_count_total",
            "command_rate",
        ],
        "network_behavior": [
            "unique_destinations",
            "unique_destination_ports",
            "unique_source_ips",
            "unique_protocols",
            "unique_honeypots",
        ],
        "credential_behavior": [
            "unique_usernames",
            "anonymous_event_count",
        ],
        "geoip_behavior": [
            "unique_countries",
            "unique_country_codes",
            "unique_cities",
            "latitude_mean",
            "longitude_mean",
        ],
        "reputation": [
            "ip_reputation_mean",
            "ip_reputation_min",
            "ip_reputation_max",
        ],
    },

    "status": "PASS",
}


with open(AUDIT, "w") as f:
    json.dump(
        audit,
        f,
        indent=2,
        default=str
    )


audit_sha = sha256_file(AUDIT)


# --------------------------------------------------------------
# [21] FINAL REPORT
# --------------------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5B-1 — COMPLETE")
print("=" * 70)

print("\nSource:")
print(SOURCE)

print("\nSource SHA:")
print(actual_sha)

print("\nOutput:")
print(OUTPUT)

print("\nOutput shape:")
print(features.shape)

print("\nOutput SHA-256:")
print(output_sha)

print("\nAudit:")
print(AUDIT)

print("\nAudit SHA-256:")
print(audit_sha)

print("\nSafety:")
print("Source modified:        NO")
print("Events preserved:       YES")
print("Sessions preserved:     YES")
print("attack_type used:       NO")
print("Label leakage:          NO")
print("ML features created:    NO")

print("\nSTATUS: PASS")
print("NEXT: STEP 5B-2 — TEMPORAL / NETWORK / GEO BEHAVIORAL FEATURES")

STEP 5B-1 — BEHAVIORAL FEATURE CONSTRUCTION

[1] SOURCE SHA-256
Expected: 7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Actual:   7447026062e36f2dd255c054a7ac321e3a73584337ded4ac5ced9f8adfdeb846
Verification: PASS

[2] SOURCE DATASET
Shape: (58895, 32)
Rows: 58895
Columns: 32

[3] REQUIRED INPUT COLUMNS
Missing: []
Required inputs: PASS
Label column detected in source: attack_stage
Label column detected in source: severity
Label column detected in source: mitre_technique
Label columns are NOT used to construct behavioral features.

[4] TIMESTAMP
Invalid timestamps: 0

[5] FEATURE CONSTRUCTION
Sessions: 11880
Behavioral features: 30

Feature columns:
01. event_count
02. session_duration_seconds
03. auth_attempt_total
04. user_attempt_total
05. pass_attempt_total
06. command_count_total
07. anonymous_event_count
08. unique_destinations
09. unique_destination_ports
10. unique_source_ips
11. unique_protocols
12. unique_honeypots
13. unique_usernames
14. unique_countries


In [ ]:
# ==============================================================
# STEP 5B-2 — TEMPORAL / NETWORK / GEO BEHAVIORAL FEATURES
# ==============================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

# --------------------------------------------------------------
# VERIFIED INPUT FROM STEP 5B-1
# --------------------------------------------------------------

SOURCE = "/content/session_behavioral_features_v1/session_behavioral_features_v1.csv"

EXPECTED_SHA = (
    "68d795a7ca76683eb66bf26dccb6c89b69711fd10bf77dc036865e1ffd712a99"
)

OUT_DIR = "/content/session_behavioral_features_v2"
os.makedirs(OUT_DIR, exist_ok=True)

OUTPUT = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v2.csv"
)

AUDIT = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v2_audit.json"
)

SHA_FILE = os.path.join(
    OUT_DIR,
    "session_behavioral_features_v2.sha256"
)


# --------------------------------------------------------------
# SHA-256
# --------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("STEP 5B-2 — TEMPORAL / NETWORK / GEO FEATURES")
print("=" * 70)


# --------------------------------------------------------------
# [1] VERIFY INPUT
# --------------------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — Step 5B-1 feature file not found:\n{SOURCE}"
    )

actual_sha = sha256_file(SOURCE)

print("\n[1] INPUT SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — Step 5B-1 feature file SHA mismatch."
    )

print("Verification: PASS")


# --------------------------------------------------------------
# [2] LOAD
# --------------------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[2] INPUT DATASET")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if len(df) != 11880:
    raise ValueError(
        f"STOP — unexpected session count: {len(df)} "
        f"(expected 11880)"
    )


# --------------------------------------------------------------
# [3] REQUIRED INPUT FEATURES
# --------------------------------------------------------------

REQUIRED = [
    "reconstructed_session_id",
    "event_count",
    "session_duration_seconds",
    "auth_attempt_total",
    "user_attempt_total",
    "pass_attempt_total",
    "command_count_total",
    "anonymous_event_count",
    "unique_destinations",
    "unique_destination_ports",
    "unique_source_ips",
    "unique_protocols",
    "unique_honeypots",
    "unique_usernames",
    "unique_countries",
    "unique_country_codes",
    "unique_cities",
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "latitude_mean",
    "longitude_mean",
    "inter_event_gap_mean",
    "inter_event_gap_min",
    "inter_event_gap_max",
    "session_start_time",
    "session_end_time",
    "session_duration_recomputed_seconds",
    "events_per_second",
    "auth_attempt_rate",
    "command_rate",
]

missing = [c for c in REQUIRED if c not in df.columns]

print("\n[3] REQUIRED FEATURES")
print("Missing:", missing)

if missing:
    raise ValueError(
        f"STOP — required Step 5B-1 features missing: {missing}"
    )

print("Required features: PASS")


# --------------------------------------------------------------
# [4] TIMESTAMP RECHECK
# --------------------------------------------------------------

df["session_start_time"] = pd.to_datetime(
    df["session_start_time"],
    errors="coerce",
    utc=True
)

df["session_end_time"] = pd.to_datetime(
    df["session_end_time"],
    errors="coerce",
    utc=True
)

invalid_start = int(df["session_start_time"].isna().sum())
invalid_end = int(df["session_end_time"].isna().sum())

print("\n[4] SESSION TIME VALIDATION")
print("Invalid start times:", invalid_start)
print("Invalid end times:", invalid_end)

if invalid_start or invalid_end:
    raise ValueError(
        "STOP — invalid session timestamps detected."
    )


# --------------------------------------------------------------
# [5] COPY — NEVER MODIFY INPUT
# --------------------------------------------------------------

features = df.copy()


# --------------------------------------------------------------
# [6] TEMPORAL FEATURES
# --------------------------------------------------------------

features["session_duration_log1p"] = np.log1p(
    features["session_duration_recomputed_seconds"].clip(lower=0)
)

features["event_count_log1p"] = np.log1p(
    features["event_count"].clip(lower=0)
)

features["inter_event_gap_mean_log1p"] = np.log1p(
    features["inter_event_gap_mean"].clip(lower=0)
)

features["inter_event_gap_max_log1p"] = np.log1p(
    features["inter_event_gap_max"].clip(lower=0)
)

features["event_density"] = np.where(
    features["session_duration_recomputed_seconds"] > 0,
    features["event_count"]
    / features["session_duration_recomputed_seconds"],
    0.0
)

features["command_density"] = np.where(
    features["event_count"] > 0,
    features["command_count_total"]
    / features["event_count"],
    0.0
)

features["auth_attempt_density"] = np.where(
    features["event_count"] > 0,
    features["auth_attempt_total"]
    / features["event_count"],
    0.0
)

features["credential_attempt_density"] = np.where(
    features["event_count"] > 0,
    (
        features["user_attempt_total"]
        + features["pass_attempt_total"]
    )
    / features["event_count"],
    0.0
)


# --------------------------------------------------------------
# [7] NETWORK BEHAVIOR
# --------------------------------------------------------------

features["destination_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_destinations"]
    / features["event_count"],
    0.0
)

features["destination_port_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_destination_ports"]
    / features["event_count"],
    0.0
)

features["protocol_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_protocols"]
    / features["event_count"],
    0.0
)

features["honeypot_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_honeypots"]
    / features["event_count"],
    0.0
)

features["username_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_usernames"]
    / features["event_count"],
    0.0
)


# --------------------------------------------------------------
# [8] GEO BEHAVIOR
# --------------------------------------------------------------

features["geo_country_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_countries"]
    / features["event_count"],
    0.0
)

features["geo_city_diversity"] = np.where(
    features["event_count"] > 0,
    features["unique_cities"]
    / features["event_count"],
    0.0
)

# Geographic magnitude, retained as a behavioral representation.
features["geo_latitude_abs"] = features[
    "latitude_mean"
].abs()

features["geo_longitude_abs"] = features[
    "longitude_mean"
].abs()


# --------------------------------------------------------------
# [9] REPUTATION BEHAVIOR
# --------------------------------------------------------------

features["ip_reputation_range"] = (
    features["ip_reputation_max"]
    - features["ip_reputation_min"]
)

features["ip_reputation_abs_mean"] = (
    features["ip_reputation_mean"].abs()
)


# --------------------------------------------------------------
# [10] SESSION TIME COMPONENTS
# --------------------------------------------------------------

features["session_start_hour"] = (
    features["session_start_time"].dt.hour
)

features["session_start_dayofweek"] = (
    features["session_start_time"].dt.dayofweek
)

features["session_start_dayofmonth"] = (
    features["session_start_time"].dt.day
)


# --------------------------------------------------------------
# [11] CONSISTENCY CHECK
# --------------------------------------------------------------

recomputed_duration = (
    features["session_end_time"]
    - features["session_start_time"]
).dt.total_seconds()

duration_difference = (
    recomputed_duration
    - features["session_duration_recomputed_seconds"]
).abs()

max_duration_difference = float(
    duration_difference.max()
)

print("\n[5] TEMPORAL CONSISTENCY")
print(
    "Maximum duration difference:",
    max_duration_difference
)

if max_duration_difference > 1e-6:
    raise ValueError(
        "STOP — session duration consistency failure."
    )


# --------------------------------------------------------------
# [12] FORBIDDEN LABEL CHECK
# --------------------------------------------------------------

FORBIDDEN = [
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique",
]

leaked = [
    c for c in FORBIDDEN
    if c in features.columns
]

print("\n[6] LABEL LEAKAGE CHECK")
print("Forbidden label columns present:", leaked)

if leaked:
    raise ValueError(
        f"STOP — forbidden label columns found: {leaked}"
    )

print("Label leakage: NO")


# --------------------------------------------------------------
# [13] NUMERIC SANITY
# --------------------------------------------------------------

numeric_cols = features.select_dtypes(
    include=[np.number]
).columns

nonfinite = {}

for col in numeric_cols:
    count = int(
        (~np.isfinite(features[col].fillna(0))).sum()
    )

    if count:
        nonfinite[col] = count

print("\n[7] NUMERIC SANITY")
print("Non-finite columns:", nonfinite)

if nonfinite:
    raise ValueError(
        f"STOP — non-finite values detected: {nonfinite}"
    )


# --------------------------------------------------------------
# [14] DUPLICATE SESSION CHECK
# --------------------------------------------------------------

duplicate_sessions = int(
    features["reconstructed_session_id"].duplicated().sum()
)

print("\n[8] SESSION UNIQUENESS")
print("Duplicate session IDs:", duplicate_sessions)

if duplicate_sessions:
    raise ValueError(
        "STOP — duplicate reconstructed sessions detected."
    )


# --------------------------------------------------------------
# [15] SAVE
# --------------------------------------------------------------

features.to_csv(
    OUTPUT,
    index=False
)

output_sha = sha256_file(OUTPUT)

with open(SHA_FILE, "w") as f:
    f.write(
        f"{output_sha}  {os.path.basename(OUTPUT)}\n"
    )


# --------------------------------------------------------------
# [16] AUDIT
# --------------------------------------------------------------

new_features = [
    c for c in features.columns
    if c not in df.columns
]

audit = {
    "step": "5B-2",
    "name": "Temporal Network Geo Behavioral Features",

    "input": {
        "path": SOURCE,
        "sha256": actual_sha,
        "rows": int(len(df)),
        "columns": int(len(df.columns)),
    },

    "output": {
        "path": OUTPUT,
        "sha256": output_sha,
        "rows": int(len(features)),
        "columns": int(len(features.columns)),
    },

    "sessions": {
        "input": int(len(df)),
        "output": int(len(features)),
        "preserved": bool(len(df) == len(features)),
        "duplicate_session_ids": duplicate_sessions,
    },

    "new_features": new_features,

    "label_safety": {
        "attack_type_used": False,
        "attack_stage_used": False,
        "severity_used": False,
        "mitre_technique_used": False,
        "label_leakage": False,
    },

    "source_protection": {
        "input_modified": False,
        "source_sha_preserved": True,
    },

    "status": "PASS",
}

with open(AUDIT, "w") as f:
    json.dump(
        audit,
        f,
        indent=2,
        default=str
    )


# --------------------------------------------------------------
# [17] FINAL REPORT
# --------------------------------------------------------------

audit_sha = sha256_file(AUDIT)

print("\n" + "=" * 70)
print("STEP 5B-2 — COMPLETE")
print("=" * 70)

print("\nInput:")
print(SOURCE)

print("\nInput SHA:")
print(actual_sha)

print("\nOutput:")
print(OUTPUT)

print("\nOutput shape:")
print(features.shape)

print("\nNew features:", len(new_features))

print("\nOutput SHA-256:")
print(output_sha)

print("\nAudit:")
print(AUDIT)

print("\nAudit SHA-256:")
print(audit_sha)

print("\nSafety:")
print("Input modified:          NO")
print("Session count preserved: YES")
print("Duplicate sessions:      NO")
print("Label leakage:           NO")
print("ML features created:     NO")

print("\nSTATUS: PASS")
print("NEXT: STEP 5C — FULL LEAKAGE + FEATURE VALIDATION AUDIT")

STEP 5B-2 — TEMPORAL / NETWORK / GEO FEATURES

[1] INPUT SHA-256
Expected: 68d795a7ca76683eb66bf26dccb6c89b69711fd10bf77dc036865e1ffd712a99
Actual:   68d795a7ca76683eb66bf26dccb6c89b69711fd10bf77dc036865e1ffd712a99
Verification: PASS

[2] INPUT DATASET
Shape: (11880, 31)
Rows: 11880
Columns: 31

[3] REQUIRED FEATURES
Missing: []
Required features: PASS

[4] SESSION TIME VALIDATION
Invalid start times: 0
Invalid end times: 0

[5] TEMPORAL CONSISTENCY
Maximum duration difference: 0.0

[6] LABEL LEAKAGE CHECK
Forbidden label columns present: []
Label leakage: NO

[7] NUMERIC SANITY
Non-finite columns: {}

[8] SESSION UNIQUENESS
Duplicate session IDs: 0

STEP 5B-2 — COMPLETE

Input:
/content/session_behavioral_features_v1/session_behavioral_features_v1.csv

Input SHA:
68d795a7ca76683eb66bf26dccb6c89b69711fd10bf77dc036865e1ffd712a99

Output:
/content/session_behavioral_features_v2/session_behavioral_features_v2.csv

Output shape:
(11880, 53)

New features: 22

Output SHA-256:
c06c4d3f08b254

In [ ]:
# ================================================================
# STEP 5C-1 — MISSING / NON-FINITE DIAGNOSTIC
# DO NOT MODIFY SOURCE
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

SOURCE = "/content/session_behavioral_features_v2/session_behavioral_features_v2.csv"
EXPECTED_SHA = "c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9"

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("=" * 70)
print("STEP 5C-1 — MISSING / NON-FINITE DIAGNOSTIC")
print("=" * 70)

# ------------------------------------------------
# 1. SOURCE
# ------------------------------------------------
if not os.path.exists(SOURCE):
    raise FileNotFoundError(f"STOP — source not found:\n{SOURCE}")

print("\n[1] SOURCE")
print("Exists: PASS")
print("Path:", SOURCE)
print("Size:", os.path.getsize(SOURCE), "bytes")

# ------------------------------------------------
# 2. SHA VERIFICATION
# ------------------------------------------------
actual_sha = sha256_file(SOURCE)

print("\n[2] SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — SHA-256 mismatch.\n"
        "The behavioral feature source is not the verified Step 5B-2 checkpoint."
    )

print("Verification: PASS")

# ------------------------------------------------
# 3. LOAD
# ------------------------------------------------
df = pd.read_csv(SOURCE, low_memory=False)

print("\n[3] DATASET")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if len(df) != 11880:
    raise ValueError(
        f"STOP — unexpected row count: {len(df)}; expected 11880."
    )

# ------------------------------------------------
# 4. SESSION ID CHECK
# ------------------------------------------------
SESSION_COL = "reconstructed_session_id"

if SESSION_COL not in df.columns:
    raise ValueError("STOP — reconstructed_session_id missing.")

missing_sessions = df[SESSION_COL].isna().sum()
duplicate_sessions = df[SESSION_COL].duplicated().sum()

print("\n[4] SESSION ID")
print("Missing:", missing_sessions)
print("Duplicate:", duplicate_sessions)
print("Unique:", df[SESSION_COL].nunique())

if missing_sessions != 0 or duplicate_sessions != 0:
    raise ValueError("STOP — session ID integrity failed.")

# ------------------------------------------------
# 5. LABEL LEAKAGE CHECK
# ------------------------------------------------
FORBIDDEN = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

present_forbidden = sorted(
    [c for c in FORBIDDEN if c in df.columns]
)

print("\n[5] LABEL LEAKAGE")
print("Forbidden label columns present:", present_forbidden)

if present_forbidden:
    raise ValueError(
        "STOP — label columns found in behavioral feature dataset."
    )

print("Label leakage: NO")

# ------------------------------------------------
# 6. MISSING VALUES
# ------------------------------------------------
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0].sort_values(
    ascending=False
)

print("\n[6] MISSING VALUES")

if len(missing_counts) == 0:
    print("Missing values: NONE")
else:
    print(missing_counts.to_string())

# ------------------------------------------------
# 7. NUMERIC NON-FINITE VALUES
# ------------------------------------------------
numeric_cols = df.select_dtypes(include=[np.number]).columns

nonfinite_report = {}

for col in numeric_cols:
    arr = pd.to_numeric(df[col], errors="coerce").to_numpy()

    count_nonfinite = int((~np.isfinite(arr)).sum())

    if count_nonfinite > 0:
        nonfinite_report[col] = count_nonfinite

print("\n[7] NON-FINITE NUMERIC VALUES")
print("Numeric columns:", len(numeric_cols))

if nonfinite_report:
    print("Non-finite columns:")
    for col, count in nonfinite_report.items():
        print(f"  {col}: {count}")
else:
    print("Non-finite values: NONE")

# ------------------------------------------------
# 8. INF vs NAN BREAKDOWN
# ------------------------------------------------
print("\n[8] NaN / +INF / -INF BREAKDOWN")

breakdown = []

for col in numeric_cols:
    s = pd.to_numeric(df[col], errors="coerce")

    nan_count = int(s.isna().sum())
    pos_inf = int(np.isposinf(s).sum())
    neg_inf = int(np.isneginf(s).sum())

    if nan_count or pos_inf or neg_inf:
        breakdown.append({
            "column": col,
            "nan": nan_count,
            "positive_inf": pos_inf,
            "negative_inf": neg_inf,
            "total_problematic": nan_count + pos_inf + neg_inf
        })

if breakdown:
    breakdown_df = pd.DataFrame(breakdown)
    print(breakdown_df.to_string(index=False))
else:
    breakdown_df = pd.DataFrame(
        columns=[
            "column",
            "nan",
            "positive_inf",
            "negative_inf",
            "total_problematic"
        ]
    )
    print("No problematic numeric values.")

# ------------------------------------------------
# 9. RANGE CHECK
# ------------------------------------------------
print("\n[9] NUMERIC RANGE SUMMARY")

range_rows = []

for col in numeric_cols:
    s = pd.to_numeric(df[col], errors="coerce")

    finite = s[np.isfinite(s)]

    if len(finite) > 0:
        range_rows.append({
            "column": col,
            "finite_count": int(len(finite)),
            "min": float(finite.min()),
            "max": float(finite.max())
        })

range_df = pd.DataFrame(range_rows)

if len(range_df):
    print(range_df.to_string(index=False))
else:
    print("No finite numeric values available.")

# ------------------------------------------------
# 10. SAVE DIAGNOSTIC ONLY
# ------------------------------------------------
OUT = "/content/session_behavioral_features_v2/session_behavioral_features_v2_missing_diagnostic.json"

diagnostic = {
    "step": "5C-1",
    "source": SOURCE,
    "source_sha256": actual_sha,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "unique_sessions": int(df[SESSION_COL].nunique()),
    "missing_values": {
        k: int(v) for k, v in missing_counts.items()
    },
    "nonfinite_values": {
        k: int(v) for k, v in nonfinite_report.items()
    },
    "nan_inf_breakdown": breakdown
}

with open(OUT, "w", encoding="utf-8") as f:
    json.dump(diagnostic, f, indent=2)

diagnostic_sha = sha256_file(OUT)

print("\n[10] DIAGNOSTIC")
print("Saved:", OUT)
print("Diagnostic SHA-256:", diagnostic_sha)

# ------------------------------------------------
# FINAL SAFETY
# ------------------------------------------------
sha_after = sha256_file(SOURCE)

print("\n[11] SOURCE PROTECTION")
print("SHA before:", actual_sha)
print("SHA after: ", sha_after)

if sha_after != actual_sha:
    raise RuntimeError("STOP — source file changed unexpectedly.")

print("Source unchanged: PASS")

print("\n" + "=" * 70)
print("STATUS: DIAGNOSTIC COMPLETE")
print("=" * 70)
print("DO NOT MODIFY session_behavioral_features_v2.csv")
print("NEXT: SEND ME THIS OUTPUT")
print("=" * 70)

STEP 5C-1 — MISSING / NON-FINITE DIAGNOSTIC

[1] SOURCE
Exists: PASS
Path: /content/session_behavioral_features_v2/session_behavioral_features_v2.csv
Size: 4757604 bytes

[2] SHA-256
Expected: c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9
Actual:   c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9
Verification: PASS

[3] DATASET
Shape: (11880, 53)
Rows: 11880
Columns: 53

[4] SESSION ID
Missing: 0
Duplicate: 0
Unique: 11880

[5] LABEL LEAKAGE
Forbidden label columns present: []
Label leakage: NO

[6] MISSING VALUES
ip_reputation_mean        11880
ip_reputation_min         11880
ip_reputation_max         11880
ip_reputation_range       11880
ip_reputation_abs_mean    11880
longitude_mean                4
latitude_mean                 4
geo_longitude_abs             4
geo_latitude_abs              4

[7] NON-FINITE NUMERIC VALUES
Numeric columns: 50
Non-finite columns:
  ip_reputation_mean: 11880
  ip_reputation_min: 11880
  ip_reputation_max: 11880
  la

In [ ]:
# ================================================================
# STEP 5C-2 — CONTROLLED MISSING-VALUE HANDLING
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

SOURCE = "/content/session_behavioral_features_v2/session_behavioral_features_v2.csv"
EXPECTED_SHA = "c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9"

OUT_DIR = "/content/session_behavioral_features_v3"
OUT = f"{OUT_DIR}/session_behavioral_features_v3.csv"
AUDIT = f"{OUT_DIR}/session_behavioral_features_v3_audit.json"
SHA_FILE = f"{OUT_DIR}/session_behavioral_features_v3.sha256"


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("STEP 5C-2 — CONTROLLED MISSING-VALUE HANDLING")
print("=" * 70)


# ------------------------------------------------
# 1. VERIFY SOURCE
# ------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — verified source not found:\n{SOURCE}"
    )

actual_sha = sha256_file(SOURCE)

print("\n[1] SOURCE SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — source SHA-256 mismatch."
    )

print("Verification: PASS")


# ------------------------------------------------
# 2. LOAD
# ------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[2] SOURCE DATASET")
print("Shape:", df.shape)

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, got {len(df)}"
    )


# ------------------------------------------------
# 3. SESSION INTEGRITY
# ------------------------------------------------

SESSION_COL = "reconstructed_session_id"

if SESSION_COL not in df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

if df[SESSION_COL].isna().any():
    raise ValueError(
        "STOP — missing reconstructed session IDs."
    )

if df[SESSION_COL].duplicated().any():
    raise ValueError(
        "STOP — duplicate reconstructed session IDs."
    )

print("\n[3] SESSION INTEGRITY")
print("Sessions:", df[SESSION_COL].nunique())
print("Missing IDs:", df[SESSION_COL].isna().sum())
print("Duplicate IDs:", df[SESSION_COL].duplicated().sum())


# ------------------------------------------------
# 4. LABEL LEAKAGE PROTECTION
# ------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

present_labels = sorted(
    c for c in FORBIDDEN_LABELS
    if c in df.columns
)

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns present:", present_labels)

if present_labels:
    raise ValueError(
        "STOP — label columns detected."
    )

print("Label leakage: NO")


# ------------------------------------------------
# 5. ORIGINAL MISSINGNESS SNAPSHOT
# ------------------------------------------------

missing_before = {
    col: int(count)
    for col, count in df.isna().sum().items()
    if count > 0
}

print("\n[5] ORIGINAL MISSING VALUES")

if missing_before:
    for col, count in missing_before.items():
        print(f"{col}: {count}")
else:
    print("None")


# ------------------------------------------------
# 6. MISSING-VALUE POLICY
# ------------------------------------------------

# IMPORTANT:
# IP reputation is completely unavailable in the source.
# We DO NOT invent a numerical reputation value.
#
# Instead:
#   NaN remains NaN for IP reputation.
#
# This preserves the semantic meaning:
# "reputation information unavailable".
#
# Geo coordinates:
# Only 4 sessions are missing.
# We use a neutral dataset-level median for the numeric
# coordinate features, and create explicit availability
# indicators so the model can distinguish imputed values
# from observed values.

GEO_COLS = [
    "latitude_mean",
    "longitude_mean",
    "geo_latitude_abs",
    "geo_longitude_abs"
]

IP_REPUTATION_COLS = [
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean"
]

# Verify expected columns
missing_geo_columns = [
    c for c in GEO_COLS
    if c not in df.columns
]

missing_ip_columns = [
    c for c in IP_REPUTATION_COLS
    if c not in df.columns
]

if missing_geo_columns:
    raise ValueError(
        f"STOP — missing Geo columns: {missing_geo_columns}"
    )

if missing_ip_columns:
    raise ValueError(
        f"STOP — missing IP reputation columns: {missing_ip_columns}"
    )


# ------------------------------------------------
# 7. GEO AVAILABILITY INDICATORS
# ------------------------------------------------

df["geo_location_available"] = (
    df["latitude_mean"].notna() &
    df["longitude_mean"].notna()
).astype("int8")


# ------------------------------------------------
# 8. GEO MEDIAN IMPUTATION
# ------------------------------------------------

geo_medians = {}

for col in GEO_COLS:

    median_value = df[col].median()

    if not np.isfinite(median_value):
        raise ValueError(
            f"STOP — cannot calculate finite median for {col}"
        )

    geo_medians[col] = float(median_value)

    df[col] = df[col].fillna(median_value)


# ------------------------------------------------
# 9. IP REPUTATION AVAILABILITY INDICATOR
# ------------------------------------------------

df["ip_reputation_available"] = (
    df["ip_reputation_mean"].notna()
).astype("int8")


# ------------------------------------------------
# 10. DO NOT IMPUTE IP REPUTATION
# ------------------------------------------------

# Keep IP reputation NaN because the source contains
# no actual reputation information for these sessions.

print("\n[6] IP REPUTATION POLICY")

for col in IP_REPUTATION_COLS:

    remaining = int(df[col].isna().sum())

    print(
        f"{col}: {remaining} missing "
        "(preserved as unavailable)"
    )


# ------------------------------------------------
# 11. POST-PROCESSING SANITY
# ------------------------------------------------

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns

nonfinite = {}

for col in numeric_cols:

    arr = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy()

    count = int(
        (~np.isfinite(arr)).sum()
    )

    if count > 0:
        nonfinite[col] = count


print("\n[7] NUMERIC SANITY")

if nonfinite:
    print("Remaining non-finite values:")

    for col, count in nonfinite.items():
        print(f"{col}: {count}")

else:
    print("No non-finite numeric values.")


# ------------------------------------------------
# 12. SAVE NEW DATASET
# ------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

df.to_csv(
    OUT,
    index=False
)

output_sha = sha256_file(OUT)

with open(SHA_FILE, "w") as f:
    f.write(
        f"{output_sha}  "
        f"{os.path.basename(OUT)}\n"
    )


# ------------------------------------------------
# 13. AUDIT
# ------------------------------------------------

missing_after = {
    col: int(count)
    for col, count in df.isna().sum().items()
    if count > 0
}

audit = {
    "step": "5C-2",
    "description": "Controlled missing-value handling",
    "source": SOURCE,
    "source_sha256": actual_sha,
    "output": OUT,
    "output_sha256": output_sha,
    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "sessions": int(df[SESSION_COL].nunique()),

    "policy": {
        "ip_reputation": (
            "preserve_missing_as_unavailable; "
            "no artificial numerical value"
        ),
        "geo_coordinates": (
            "median imputation for 4 affected sessions "
            "with explicit availability indicator"
        )
    },

    "geo_medians": geo_medians,

    "missing_before": missing_before,
    "missing_after": missing_after,

    "added_features": [
        "geo_location_available",
        "ip_reputation_available"
    ],

    "label_columns_used": [],

    "source_modified": False
}

with open(
    AUDIT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(AUDIT)


# ------------------------------------------------
# 14. SOURCE PROTECTION
# ------------------------------------------------

source_sha_after = sha256_file(SOURCE)

print("\n[8] SOURCE PROTECTION")

print("Before:", actual_sha)
print("After: ", source_sha_after)

if source_sha_after != actual_sha:
    raise RuntimeError(
        "STOP — SOURCE FILE CHANGED."
    )

print("Source unchanged: PASS")


# ------------------------------------------------
# 15. FINAL REPORT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5C-2 — COMPLETE")
print("=" * 70)

print("\nInput:")
print(SOURCE)

print("\nInput SHA:")
print(actual_sha)

print("\nOutput:")
print(OUT)

print("Output shape:")
print(df.shape)

print("\nOutput SHA-256:")
print(output_sha)

print("\nAudit:")
print(AUDIT)

print("Audit SHA-256:")
print(audit_sha)

print("\nSHA manifest:")
print(SHA_FILE)

print("\nSafety:")
print("Source modified:        NO")
print("Sessions preserved:     YES")
print("Labels used:            NO")
print("IP reputation invented: NO")
print("Geo availability tracked: YES")

print("\nSTATUS: PASS")
print("NEXT: STEP 5C-3 — FINAL FEATURE LEAKAGE + FINITE-VALUE AUDIT")
print("=" * 70)

STEP 5C-2 — CONTROLLED MISSING-VALUE HANDLING

[1] SOURCE SHA-256
Expected: c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9
Actual:   c06c4d3f08b2542059c01cb2f85c13e16fa4356059e7b3523a875bb0634abee9
Verification: PASS

[2] SOURCE DATASET
Shape: (11880, 53)

[3] SESSION INTEGRITY
Sessions: 11880
Missing IDs: 0
Duplicate IDs: 0

[4] LABEL LEAKAGE
Forbidden columns present: []
Label leakage: NO

[5] ORIGINAL MISSING VALUES
ip_reputation_mean: 11880
ip_reputation_min: 11880
ip_reputation_max: 11880
latitude_mean: 4
longitude_mean: 4
geo_latitude_abs: 4
geo_longitude_abs: 4
ip_reputation_range: 11880
ip_reputation_abs_mean: 11880

[6] IP REPUTATION POLICY
ip_reputation_mean: 11880 missing (preserved as unavailable)
ip_reputation_min: 11880 missing (preserved as unavailable)
ip_reputation_max: 11880 missing (preserved as unavailable)
ip_reputation_range: 11880 missing (preserved as unavailable)
ip_reputation_abs_mean: 11880 missing (preserved as unavailable)

[7] NUMERIC SAN

In [ ]:
# ================================================================
# STEP 5C-3 — FINAL ML-SAFE FEATURE VALIDATION
#              + EXPLICIT MISSINGNESS REPRESENTATION
# ================================================================

import os
import json
import hashlib
import numpy as np
import pandas as pd

SOURCE = "/content/session_behavioral_features_v3/session_behavioral_features_v3.csv"
EXPECTED_SHA = "c8bce0f655166f848e355242f2ddbc7ca639fb34b53ea7a044fbd1e1f507142c"

OUT_DIR = "/content/session_behavioral_features_v4"
OUT = f"{OUT_DIR}/session_behavioral_features_v4.csv"
AUDIT = f"{OUT_DIR}/session_behavioral_features_v4_audit.json"
SHA_FILE = f"{OUT_DIR}/session_behavioral_features_v4.sha256"


def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("STEP 5C-3 — FINAL ML-SAFE FEATURE VALIDATION")
print("=" * 70)


# ------------------------------------------------
# 1. SOURCE VERIFICATION
# ------------------------------------------------

if not os.path.exists(SOURCE):
    raise FileNotFoundError(
        f"STOP — verified v3 dataset not found:\n{SOURCE}"
    )

actual_sha = sha256_file(SOURCE)

print("\n[1] SOURCE SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:
    raise ValueError(
        "STOP — v3 SHA-256 mismatch."
    )

print("Verification: PASS")


# ------------------------------------------------
# 2. LOAD
# ------------------------------------------------

df = pd.read_csv(SOURCE, low_memory=False)

print("\n[2] DATASET")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))

if len(df) != 11880:
    raise ValueError(
        f"STOP — expected 11880 sessions, got {len(df)}"
    )


# ------------------------------------------------
# 3. SESSION INTEGRITY
# ------------------------------------------------

SESSION_COL = "reconstructed_session_id"

if SESSION_COL not in df.columns:
    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

missing_sessions = int(df[SESSION_COL].isna().sum())
duplicate_sessions = int(df[SESSION_COL].duplicated().sum())
unique_sessions = int(df[SESSION_COL].nunique())

print("\n[3] SESSION INTEGRITY")
print("Unique sessions:", unique_sessions)
print("Missing IDs:", missing_sessions)
print("Duplicate IDs:", duplicate_sessions)

if missing_sessions != 0 or duplicate_sessions != 0:
    raise ValueError(
        "STOP — session integrity failure."
    )


# ------------------------------------------------
# 4. LABEL LEAKAGE
# ------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

forbidden_present = sorted(
    c for c in FORBIDDEN_LABELS
    if c in df.columns
)

print("\n[4] LABEL LEAKAGE")
print("Forbidden columns:", forbidden_present)

if forbidden_present:
    raise ValueError(
        f"STOP — label columns present: {forbidden_present}"
    )

print("Label leakage: NO")


# ------------------------------------------------
# 5. IP REPUTATION — EXPLICIT UNAVAILABLE STATE
# ------------------------------------------------

IP_REPUTATION_COLS = [
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean"
]

missing_ip_cols = [
    c for c in IP_REPUTATION_COLS
    if c not in df.columns
]

if missing_ip_cols:
    raise ValueError(
        f"STOP — missing IP reputation columns: {missing_ip_cols}"
    )

print("\n[5] IP REPUTATION AVAILABILITY")

ip_missing_before = {}

for col in IP_REPUTATION_COLS:
    count = int(df[col].isna().sum())
    ip_missing_before[col] = count
    print(f"{col}: {count} missing")

# All five are unavailable for every session.
# We do NOT invent numerical reputation values.
#
# Instead:
#   1. Keep the original columns untouched.
#   2. Add one explicit binary availability feature.
#   3. Add one explicit missingness count feature.
#
# The original NaNs are then excluded from the numeric ML matrix
# by the later preprocessing pipeline rather than being silently
# converted into fake reputation scores.

df["ip_reputation_missing_count"] = (
    df[IP_REPUTATION_COLS].isna().sum(axis=1).astype("int8")
)

df["ip_reputation_available"] = (
    df["ip_reputation_missing_count"] == 0
).astype("int8")

print(
    "All-session unavailable state:",
    bool(
        (df["ip_reputation_missing_count"] == len(IP_REPUTATION_COLS)).all()
    )
)


# ------------------------------------------------
# 6. GEO VALIDATION
# ------------------------------------------------

GEO_COLS = [
    "latitude_mean",
    "longitude_mean",
    "geo_latitude_abs",
    "geo_longitude_abs"
]

print("\n[6] GEO VALIDATION")

for col in GEO_COLS:

    if col not in df.columns:
        raise ValueError(
            f"STOP — missing Geo feature: {col}"
        )

    missing = int(df[col].isna().sum())
    print(f"{col}: {missing} missing")

    if missing != 0:
        raise ValueError(
            f"STOP — Geo feature still contains missing values: {col}"
        )

print("Geo features: PASS")


# ------------------------------------------------
# 7. NUMERIC FINITE CHECK
# ------------------------------------------------

print("\n[7] NUMERIC FINITE CHECK")

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

nonfinite = {}

for col in numeric_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy()

    count = int(
        (~np.isfinite(values)).sum()
    )

    if count > 0:
        nonfinite[col] = count

if nonfinite:
    print("Non-finite values detected:")

    for col, count in nonfinite.items():
        print(f"{col}: {count}")

    # The only allowed non-finite values are the intentionally
    # preserved IP-reputation unavailable values.
    allowed = set(IP_REPUTATION_COLS)

    unexpected = {
        col: count
        for col, count in nonfinite.items()
        if col not in allowed
    }

    if unexpected:
        raise ValueError(
            f"STOP — unexpected non-finite values: {unexpected}"
        )

    print(
        "Only documented IP reputation missingness remains: PASS"
    )

else:
    print("No non-finite values: PASS")


# ------------------------------------------------
# 8. FEATURE INVENTORY
# ------------------------------------------------

print("\n[8] FEATURE INVENTORY")

non_feature_columns = {
    SESSION_COL,
    "session_start_time",
    "session_end_time"
}

feature_columns = [
    c for c in df.columns
    if c not in non_feature_columns
]

# Ensure forbidden labels cannot sneak into the feature set.

leakage_in_features = sorted(
    set(feature_columns) & FORBIDDEN_LABELS
)

if leakage_in_features:
    raise ValueError(
        f"STOP — labels found in feature set: {leakage_in_features}"
    )

print("Feature columns:", len(feature_columns))
print("Leakage columns:", leakage_in_features)


# ------------------------------------------------
# 9. OUTPUT
# ------------------------------------------------

os.makedirs(OUT_DIR, exist_ok=True)

df.to_csv(
    OUT,
    index=False
)

output_sha = sha256_file(OUT)

with open(SHA_FILE, "w") as f:
    f.write(
        f"{output_sha}  {os.path.basename(OUT)}\n"
    )


# ------------------------------------------------
# 10. FINAL AUDIT
# ------------------------------------------------

audit = {
    "step": "5C-3",
    "description": (
        "Final behavioral feature validation and "
        "explicit missingness representation"
    ),

    "source": SOURCE,
    "source_sha256": actual_sha,

    "output": OUT,
    "output_sha256": output_sha,

    "rows": int(len(df)),
    "columns": int(len(df.columns)),
    "sessions": unique_sessions,

    "label_columns_used": [],

    "ip_reputation_policy": {
        "original_values_modified": False,
        "missing_values_imputed": False,
        "availability_feature_added": True,
        "missing_count_feature_added": True,
        "reason": (
            "IP reputation is unavailable for every session; "
            "no artificial numerical reputation value was introduced."
        )
    },

    "ip_reputation_missing_before": ip_missing_before,

    "geo_policy": {
        "missing_values_remaining": 0,
        "reason": (
            "Geo missingness was handled in Step 5C-2 "
            "with explicit availability tracking."
        )
    },

    "nonfinite_policy": {
        "remaining_nonfinite": {
            col: int(count)
            for col, count in nonfinite.items()
        },
        "unexpected_nonfinite": {}
    },

    "source_modified": False
}

with open(
    AUDIT,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        audit,
        f,
        indent=2
    )

audit_sha = sha256_file(AUDIT)


# ------------------------------------------------
# 11. SOURCE PROTECTION
# ------------------------------------------------

source_sha_after = sha256_file(SOURCE)

print("\n[9] SOURCE PROTECTION")
print("Before:", actual_sha)
print("After: ", source_sha_after)

if source_sha_after != actual_sha:
    raise RuntimeError(
        "STOP — source dataset changed."
    )

print("Source unchanged: PASS")


# ------------------------------------------------
# 12. FINAL REPORT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5C-3 — COMPLETE")
print("=" * 70)

print("\nInput:")
print(SOURCE)

print("\nInput SHA:")
print(actual_sha)

print("\nOutput:")
print(OUT)

print("\nOutput shape:")
print(df.shape)

print("\nOutput SHA-256:")
print(output_sha)

print("\nAudit:")
print(AUDIT)

print("\nAudit SHA-256:")
print(audit_sha)

print("\nSHA manifest:")
print(SHA_FILE)

print("\nSafety:")
print("Source modified:              NO")
print("Sessions preserved:           YES")
print("Labels used:                  NO")
print("Artificial IP reputation:     NO")
print("Geo missingness unresolved:   NO")
print("Missingness explicitly tracked: YES")

print("\nSTATUS: PASS")
print("NEXT: STEP 5C-4 — FINAL PRE-ML AUDIT + CHECKPOINT")
print("=" * 70)

STEP 5C-3 — FINAL ML-SAFE FEATURE VALIDATION

[1] SOURCE SHA-256
Expected: c8bce0f655166f848e355242f2ddbc7ca639fb34b53ea7a044fbd1e1f507142c
Actual:   c8bce0f655166f848e355242f2ddbc7ca639fb34b53ea7a044fbd1e1f507142c
Verification: PASS

[2] DATASET
Shape: (11880, 55)
Rows: 11880
Columns: 55

[3] SESSION INTEGRITY
Unique sessions: 11880
Missing IDs: 0
Duplicate IDs: 0

[4] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[5] IP REPUTATION AVAILABILITY
ip_reputation_mean: 11880 missing
ip_reputation_min: 11880 missing
ip_reputation_max: 11880 missing
ip_reputation_range: 11880 missing
ip_reputation_abs_mean: 11880 missing
All-session unavailable state: True

[6] GEO VALIDATION
latitude_mean: 0 missing
longitude_mean: 0 missing
geo_latitude_abs: 0 missing
geo_longitude_abs: 0 missing
Geo features: PASS

[7] NUMERIC FINITE CHECK
Non-finite values detected:
ip_reputation_mean: 11880
ip_reputation_min: 11880
ip_reputation_max: 11880
ip_reputation_range: 11880
ip_reputation_abs_mean: 1188

In [ ]:
# ================================================================
# STEP 5C-4 — FINAL PRE-ML AUDIT + CHECKPOINT
# ================================================================

import os
import json
import hashlib
import shutil
from datetime import datetime, timezone

SOURCE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.csv"
SOURCE_SHA_FILE = "/content/session_behavioral_features_v4/session_behavioral_features_v4.sha256"
SOURCE_AUDIT = "/content/session_behavioral_features_v4/session_behavioral_features_v4_audit.json"

EXPECTED_SHA = "b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534"

CHECKPOINT_DIR = "/content/checkpoints/session_behavioral_features_v4_checkpoint"

BACKUP_DIR = os.path.join(
    CHECKPOINT_DIR,
    "backup"
)

CHECKPOINT_AUDIT = os.path.join(
    CHECKPOINT_DIR,
    "session_behavioral_features_v4_checkpoint_audit.json"
)

CHECKPOINT_SHA = os.path.join(
    CHECKPOINT_DIR,
    "session_behavioral_features_v4_checkpoint.sha256"
)

BACKUP_MANIFEST = os.path.join(
    CHECKPOINT_DIR,
    "backup_manifest.json"
)

BACKUP_MANIFEST_SHA = os.path.join(
    CHECKPOINT_DIR,
    "backup_manifest.sha256"
)


def sha256_file(path, chunk_size=1024 * 1024):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


print("=" * 70)
print("STEP 5C-4 — FINAL PRE-ML AUDIT + CHECKPOINT")
print("=" * 70)


# ------------------------------------------------
# 1. REQUIRED FILES
# ------------------------------------------------

required = [
    SOURCE,
    SOURCE_SHA_FILE,
    SOURCE_AUDIT
]

print("\n[1] REQUIRED FILES")

for path in required:

    status = os.path.exists(path)

    print(
        os.path.basename(path),
        ":",
        "PASS" if status else "MISSING"
    )

    if not status:
        raise FileNotFoundError(
            f"STOP — required file missing:\n{path}"
        )


# ------------------------------------------------
# 2. SOURCE SHA
# ------------------------------------------------

actual_sha = sha256_file(SOURCE)

print("\n[2] SOURCE SHA-256")
print("Expected:", EXPECTED_SHA)
print("Actual:  ", actual_sha)

if actual_sha != EXPECTED_SHA:

    raise ValueError(
        "STOP — behavioral feature checkpoint SHA mismatch."
    )

print("Verification: PASS")


# ------------------------------------------------
# 3. LOAD DATASET
# ------------------------------------------------

import pandas as pd
import numpy as np

df = pd.read_csv(
    SOURCE,
    low_memory=False
)

print("\n[3] DATASET")
print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))


if len(df) != 11880:

    raise ValueError(
        f"STOP — expected 11880 sessions, got {len(df)}"
    )


# ------------------------------------------------
# 4. SESSION INTEGRITY
# ------------------------------------------------

SESSION_COL = "reconstructed_session_id"

if SESSION_COL not in df.columns:

    raise ValueError(
        "STOP — reconstructed_session_id missing."
    )

missing_ids = int(
    df[SESSION_COL].isna().sum()
)

duplicate_ids = int(
    df[SESSION_COL].duplicated().sum()
)

unique_ids = int(
    df[SESSION_COL].nunique()
)

print("\n[4] SESSION INTEGRITY")
print("Unique sessions:", unique_ids)
print("Missing IDs:", missing_ids)
print("Duplicate IDs:", duplicate_ids)

if missing_ids != 0:
    raise ValueError(
        "STOP — missing session IDs."
    )

if duplicate_ids != 0:
    raise ValueError(
        "STOP — duplicate session IDs."
    )

if unique_ids != 11880:
    raise ValueError(
        "STOP — session count changed."
    )


# ------------------------------------------------
# 5. LABEL LEAKAGE
# ------------------------------------------------

FORBIDDEN_LABELS = {
    "attack_type",
    "attack_stage",
    "severity",
    "mitre_technique"
}

labels_present = sorted(
    set(df.columns) & FORBIDDEN_LABELS
)

print("\n[5] LABEL LEAKAGE")
print("Forbidden columns:", labels_present)

if labels_present:

    raise ValueError(
        f"STOP — label columns detected: {labels_present}"
    )

print("Label leakage: NO")


# ------------------------------------------------
# 6. REQUIRED BEHAVIORAL FEATURES
# ------------------------------------------------

REQUIRED_FEATURES = [

    "event_count",
    "session_duration_seconds",
    "auth_attempt_total",
    "user_attempt_total",
    "pass_attempt_total",
    "command_count_total",
    "anonymous_event_count",
    "unique_destinations",
    "unique_destination_ports",
    "unique_source_ips",
    "unique_protocols",
    "unique_honeypots",
    "unique_usernames",
    "unique_countries",
    "unique_country_codes",
    "unique_cities",

    "session_duration_recomputed_seconds",
    "events_per_second",
    "auth_attempt_rate",
    "command_rate",

    "session_duration_log1p",
    "event_count_log1p",
    "inter_event_gap_mean_log1p",
    "inter_event_gap_max_log1p",

    "event_density",
    "command_density",
    "auth_attempt_density",
    "credential_attempt_density",

    "destination_diversity",
    "destination_port_diversity",
    "protocol_diversity",
    "honeypot_diversity",
    "username_diversity",
    "geo_country_diversity",
    "geo_city_diversity",

    "geo_latitude_abs",
    "geo_longitude_abs",

    "ip_reputation_range",
    "ip_reputation_abs_mean",

    "session_start_hour",
    "session_start_dayofweek",
    "session_start_dayofmonth",

    "ip_reputation_missing_count",
    "ip_reputation_available"
]

missing_features = [
    c for c in REQUIRED_FEATURES
    if c not in df.columns
]

print("\n[6] FEATURE INVENTORY")
print("Required behavioral features:", len(REQUIRED_FEATURES))
print("Missing:", missing_features)

if missing_features:

    raise ValueError(
        f"STOP — required behavioral features missing: {missing_features}"
    )

print("Feature inventory: PASS")


# ------------------------------------------------
# 7. IP REPUTATION POLICY
# ------------------------------------------------

IP_COLS = [
    "ip_reputation_mean",
    "ip_reputation_min",
    "ip_reputation_max",
    "ip_reputation_range",
    "ip_reputation_abs_mean"
]

print("\n[7] IP REPUTATION POLICY")

for col in IP_COLS:

    missing = int(df[col].isna().sum())

    print(
        f"{col}: {missing} missing"
    )

    if missing != 11880:

        raise ValueError(
            f"STOP — unexpected IP reputation availability in {col}"
        )


availability_values = set(
    df["ip_reputation_available"].dropna().unique()
)

if availability_values != {0}:

    raise ValueError(
        "STOP — IP reputation availability indicator is inconsistent."
    )

print("IP reputation policy: PASS")
print("No artificial reputation values introduced.")


# ------------------------------------------------
# 8. GEO VALIDATION
# ------------------------------------------------

GEO_FEATURES = [
    "latitude_mean",
    "longitude_mean",
    "geo_latitude_abs",
    "geo_longitude_abs"
]

print("\n[8] GEO VALIDATION")

for col in GEO_FEATURES:

    missing = int(df[col].isna().sum())

    print(
        f"{col}: {missing} missing"
    )

    if missing != 0:

        raise ValueError(
            f"STOP — missing Geo feature: {col}"
        )

print("Geo validation: PASS")


# ------------------------------------------------
# 9. NUMERIC SANITY
# ------------------------------------------------

print("\n[9] NUMERIC SANITY")

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()

unexpected_nonfinite = {}

for col in numeric_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy()

    nonfinite_count = int(
        (~np.isfinite(values)).sum()
    )

    if nonfinite_count > 0:

        if col not in IP_COLS:

            unexpected_nonfinite[col] = nonfinite_count

if unexpected_nonfinite:

    raise ValueError(
        "STOP — unexpected non-finite values:\n"
        + str(unexpected_nonfinite)
    )

print(
    "Non-finite values outside documented IP reputation:",
    0
)

print("Numeric sanity: PASS")


# ------------------------------------------------
# 10. DUPLICATE FEATURE ROWS
# ------------------------------------------------

feature_columns = [
    c for c in df.columns
    if c != SESSION_COL
]

duplicate_feature_rows = int(
    df.duplicated(
        subset=feature_columns
    ).sum()
)

print("\n[10] DUPLICATE FEATURE ROWS")
print(
    "Duplicate feature rows:",
    duplicate_feature_rows
)

# Duplicate behavioral profiles are allowed.
# They are not automatically errors.
print(
    "Duplicate behavioral profiles are retained."
)


# ------------------------------------------------
# 11. FEATURE / LABEL SEPARATION
# ------------------------------------------------

feature_label_overlap = sorted(
    set(df.columns) & FORBIDDEN_LABELS
)

print("\n[11] FEATURE / LABEL SEPARATION")

if feature_label_overlap:

    raise ValueError(
        f"STOP — label leakage detected: {feature_label_overlap}"
    )

print("Feature/label separation: PASS")


# ------------------------------------------------
# 12. CREATE CHECKPOINT
# ------------------------------------------------

os.makedirs(BACKUP_DIR, exist_ok=True)

backup_dataset = os.path.join(
    BACKUP_DIR,
    "session_behavioral_features_v4.csv"
)

backup_sha = os.path.join(
    BACKUP_DIR,
    "session_behavioral_features_v4.sha256"
)

backup_audit = os.path.join(
    BACKUP_DIR,
    "session_behavioral_features_v4_audit.json"
)

backup_dataset_audit = os.path.join(
    BACKUP_DIR,
    "session_behavioral_features_v4_checkpoint_audit.json"
)

shutil.copy2(
    SOURCE,
    backup_dataset
)

shutil.copy2(
    SOURCE_SHA_FILE,
    backup_sha
)

shutil.copy2(
    SOURCE_AUDIT,
    backup_audit
)


# ------------------------------------------------
# 13. VERIFY BACKUP
# ------------------------------------------------

backup_dataset_sha = sha256_file(
    backup_dataset
)

backup_source_sha = sha256_file(
    SOURCE
)

print("\n[12] BACKUP VERIFICATION")

print(
    "Original SHA:",
    backup_source_sha
)

print(
    "Backup SHA:  ",
    backup_dataset_sha
)

if backup_dataset_sha != backup_source_sha:

    raise RuntimeError(
        "STOP — backup SHA mismatch."
    )

print("Backup dataset verification: PASS")


# ------------------------------------------------
# 14. CHECKPOINT AUDIT
# ------------------------------------------------

checkpoint_audit_data = {

    "step": "5C-4",

    "description":
        "Final pre-ML behavioral feature audit and checkpoint",

    "source":
        SOURCE,

    "source_sha256":
        actual_sha,

    "rows":
        int(len(df)),

    "columns":
        int(len(df.columns)),

    "unique_sessions":
        unique_ids,

    "missing_session_ids":
        missing_ids,

    "duplicate_session_ids":
        duplicate_ids,

    "label_columns_present":
        labels_present,

    "required_features_missing":
        missing_features,

    "ip_reputation_policy":
        "unavailable; no artificial values introduced",

    "ip_reputation_available_all_sessions":
        False,

    "geo_missing_values":
        0,

    "unexpected_nonfinite_values":
        unexpected_nonfinite,

    "duplicate_feature_rows":
        duplicate_feature_rows,

    "source_modified":
        False,

    "checkpoint_created":
        True,

    "created_utc":
        datetime.now(timezone.utc).isoformat()
}

with open(
    CHECKPOINT_AUDIT,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        checkpoint_audit_data,
        f,
        indent=2
    )


# ------------------------------------------------
# 15. CHECKPOINT SHA
# ------------------------------------------------

checkpoint_audit_sha = sha256_file(
    CHECKPOINT_AUDIT
)

with open(
    CHECKPOINT_SHA,
    "w"
) as f:

    f.write(
        f"{checkpoint_audit_sha}  "
        f"{os.path.basename(CHECKPOINT_AUDIT)}\n"
    )


# ------------------------------------------------
# 16. BACKUP MANIFEST
# ------------------------------------------------

backup_files = [
    backup_dataset,
    backup_sha,
    backup_audit,
    CHECKPOINT_AUDIT,
    CHECKPOINT_SHA
]

manifest = {

    "checkpoint":
        "session_behavioral_features_v4",

    "files": {}

}

for path in backup_files:

    manifest["files"][
        os.path.basename(path)
    ] = {
        "size_bytes": os.path.getsize(path),
        "sha256": sha256_file(path)
    }

with open(
    BACKUP_MANIFEST,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        manifest,
        f,
        indent=2
    )


manifest_sha = sha256_file(
    BACKUP_MANIFEST
)

with open(
    BACKUP_MANIFEST_SHA,
    "w"
) as f:

    f.write(
        f"{manifest_sha}  "
        f"{os.path.basename(BACKUP_MANIFEST)}\n"
    )


# ------------------------------------------------
# 17. FINAL SOURCE PROTECTION
# ------------------------------------------------

final_source_sha = sha256_file(
    SOURCE
)

if final_source_sha != actual_sha:

    raise RuntimeError(
        "STOP — source changed during checkpoint."
    )


# ------------------------------------------------
# FINAL REPORT
# ------------------------------------------------

print("\n" + "=" * 70)
print("STEP 5C-4 — COMPLETE")
print("=" * 70)

print("\nSource:")
print(SOURCE)

print("\nSource SHA:")
print(actual_sha)

print("\nCheckpoint:")
print(CHECKPOINT_DIR)

print("\nBackup files:")
for path in backup_files:
    print(
        "PASS:",
        os.path.basename(path)
    )

print("\nBackup manifest:")
print(BACKUP_MANIFEST)

print("\nBackup manifest SHA-256:")
print(manifest_sha)

print("\nSafety:")
print("Source modified:              NO")
print("Sessions preserved:           YES")
print("Session IDs preserved:        YES")
print("Labels used:                  NO")
print("Label leakage:                NO")
print("Artificial IP reputation:     NO")
print("Geo missingness:              0")
print("Unexpected non-finite:        0")
print("Checkpoint backup:            PASS")
print("Backup SHA verification:      PASS")

print("\nSTATUS: PASS")
print("STEP 5 COMPLETE")
print("NEXT: STEP 6 — UNSUPERVISED ML")
print("=" * 70)

STEP 5C-4 — FINAL PRE-ML AUDIT + CHECKPOINT

[1] REQUIRED FILES
session_behavioral_features_v4.csv : PASS
session_behavioral_features_v4.sha256 : PASS
session_behavioral_features_v4_audit.json : PASS

[2] SOURCE SHA-256
Expected: b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Actual:   b9a5cdfd2dcc3369e5831ed4c3306f548fe48d951f216f15e71967de1e418534
Verification: PASS

[3] DATASET
Shape: (11880, 56)
Rows: 11880
Columns: 56

[4] SESSION INTEGRITY
Unique sessions: 11880
Missing IDs: 0
Duplicate IDs: 0

[5] LABEL LEAKAGE
Forbidden columns: []
Label leakage: NO

[6] FEATURE INVENTORY
Required behavioral features: 44
Missing: []
Feature inventory: PASS

[7] IP REPUTATION POLICY
ip_reputation_mean: 11880 missing
ip_reputation_min: 11880 missing
ip_reputation_max: 11880 missing
ip_reputation_range: 11880 missing
ip_reputation_abs_mean: 11880 missing
IP reputation policy: PASS
No artificial reputation values introduced.

[8] GEO VALIDATION
latitude_mean: 0 missing
longitude_m

In [ ]:
# ============================================================
# STEP 5 — DOWNLOAD VERIFIED FEATURE DATASET + AUDITS
# ============================================================

import os
import zipfile
from google.colab import files

# ------------------------------------------------------------
# VERIFIED SOURCE
# ------------------------------------------------------------

SOURCE_DIR = "/content/session_behavioral_features_v4"
ZIP_PATH = "/content/STEP_5_BEHAVIORAL_FEATURES_v4_FILES.zip"

required_files = [
    "session_behavioral_features_v4.csv",
    "session_behavioral_features_v4.sha256",
    "session_behavioral_features_v4_audit.json",
]

# ------------------------------------------------------------
# 1. Check files
# ------------------------------------------------------------

missing = []

for name in required_files:
    path = os.path.join(SOURCE_DIR, name)

    if not os.path.isfile(path):
        missing.append(name)

if missing:
    raise FileNotFoundError(
        "STOP — Step 5 v4 files are missing.\n"
        f"Expected directory: {SOURCE_DIR}\n"
        f"Missing files: {missing}"
    )

# ------------------------------------------------------------
# 2. Create ZIP
# ------------------------------------------------------------

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for name in required_files:
        path = os.path.join(SOURCE_DIR, name)
        z.write(path, arcname=name)

# ------------------------------------------------------------
# 3. Verify package
# ------------------------------------------------------------

if not os.path.isfile(ZIP_PATH):
    raise FileNotFoundError(
        "STOP — ZIP package was not created."
    )

print("=" * 70)
print("STEP 5 — VERIFIED FEATURE DATASET DOWNLOAD")
print("=" * 70)

print("\nSource directory:")
print(SOURCE_DIR)

print("\nFiles included:")

for name in required_files:
    path = os.path.join(SOURCE_DIR, name)
    print(f"PASS: {name}")
    print(f"      Size: {os.path.getsize(path):,} bytes")

print("\nZIP package:")
print(ZIP_PATH)

print("\nSTATUS: READY TO DOWNLOAD")

# ------------------------------------------------------------
# 4. Download
# ------------------------------------------------------------

files.download(ZIP_PATH)

STEP 5 — VERIFIED FEATURE DATASET DOWNLOAD

Source directory:
/content/session_behavioral_features_v4

Files included:
PASS: session_behavioral_features_v4.csv
      Size: 4,814,141 bytes
PASS: session_behavioral_features_v4.sha256
      Size: 101 bytes
PASS: session_behavioral_features_v4_audit.json
      Size: 1,525 bytes

ZIP package:
/content/STEP_5_BEHAVIORAL_FEATURES_v4_FILES.zip

STATUS: READY TO DOWNLOAD


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>